# Submission: mlp_4out_emb before tuning, with step decay

Report: [mlp_report.md](../../mlp_report.md#final-comparison)


In [1]:
import os
import sys

sys.path.insert(0, "../../..")  # the ashrae package lives in the homework folder
sys.path.insert(0, "../..")  # the section's src package
from ashrae.config import PROCESSED
from ashrae.data import read_processed
from ashrae.features import build_features
from ashrae.submission import predict_chunked, save_submission
from src.data import make_loaders
from src.networks import MeterMLP
from src.optim import step_decay
from src.predict import predict_rows
from src.tracking import Experiment

SMOKE = os.environ.get("SMOKE") == "1"
CHUNK_SIZE = 1_000_000  # test rows predicted at once
EMB_DIM = 16  # exp02, before exp03 picked 12
LR = 1e-3  # exp02; divided by 10 at half and three quarters of training, as in exp17
if SMOKE:
    EXPERIMENT = "ashrae-submission-smoke"
    N_TRAIN = 20_000
    N_TEST = 50_000
    EPOCHS = 4  # long enough for both lr drops
    SUBMISSION_PATH = None  # partial test set, so nothing to submit
else:
    EXPERIMENT = "ashrae-submission"
    N_TRAIN = 2_000_000  # the same row count as the other submissions, spread over all of 2016
    N_TEST = None  # all 41.7M test rows
    EPOCHS = 10  # as in exp02
    SUBMISSION_PATH = PROCESSED / "submission_mlp_4out_emb_step.csv"

experiment = Experiment(EXPERIMENT, "submission_mlp_step")

/Users/lupaskodmitro/FuckIT/Machine Learning/Deep Learning Homeworks/Homework 1 MLP/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MLflow: sqlite:////Users/lupaskodmitro/FuckIT/Machine Learning/Deep Learning Homeworks/Homework 1 MLP/mlflow.db, experiment 'ashrae-submission', session 2026-10-13 12:06:40


## Training rows


In [2]:
df = read_processed()
n_buildings = int(df.building_id.max()) + 1
train_df = df[~df.anomaly].sample(
    N_TRAIN, random_state=0
)  # all of 2016, anomalies dropped
del df

year_median = train_df.year_built.median()  # reused for the test set
X_train = build_features(train_df, year_median)
train_loader, _, scaler = make_loaders(train_df, None, X_train, None)
print(f"{len(train_df):,} training rows, {X_train.shape[1]} features")

2,000,000 training rows, 33 features


## Model


In [3]:
# no BatchNorm or dropout: the defaults of MeterMLP
model, run_id = experiment.fit(
    lambda: MeterMLP(X_train.shape[1], n_buildings, emb_dim=EMB_DIM),
    train_loader,
    run_name="mlp_4out_emb | step | all 2016",
    epochs=EPOCHS,
    lr=LR,
    optimizer=step_decay(LR, EPOCHS),
    params={"n_train": N_TRAIN, "emb_dim": EMB_DIM, "schedule": "step"},
    tags={"model": "mlp_4out_emb", "schedule": "step"},
)
model = model.eval()
print(
    f"mlp_4out_emb, step decay: training RMSLE {experiment.history(run_id, 'train_rmsle')[-1]:.3f}"
)

/Users/lupaskodmitro/FuckIT/Machine Learning/Deep Learning Homeworks/Homework 1 MLP/.venv/lib/python3.14/site-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
mlp_4out_emb | step | all 2016: 100%|██████████| 4890/4890 [00:19<00:00, 250.84batch/s]

mlp_4out_emb, step decay: training RMSLE 0.860


## Test predictions


In [4]:
test_df = read_processed("test_merged.parquet")
if N_TEST:
    test_df = test_df.head(N_TEST)

mlp_log = predict_chunked(
    lambda rows: predict_rows(model, rows, scaler, X_train.columns, year_median),
    test_df,
    CHUNK_SIZE,
)
print(f"mean log1p prediction: {mlp_log.mean():.3f}")

mean log1p prediction: 4.238


In [5]:
save_submission(mlp_log, test_df, SUBMISSION_PATH)

saved /Users/lupaskodmitro/FuckIT/Machine Learning/Deep Learning Homeworks/Homework 1 MLP/data/processed/submission_mlp_4out_emb_step.csv: 41,697,600 rows


,meter_reading
count,41697600.00
mean,418.55
std,2521.75
min,0.00
25%,21.53
50%,78.39
75%,250.80
max,259807.97


In [6]:
if SUBMISSION_PATH:  # multithreaded 7-Zip CLI (brew install sevenzip), much faster than py7zr
    !cd "{SUBMISSION_PATH.parent}" && rm -f "{SUBMISSION_PATH.stem}.7z" && 7zz a -mmt=on "{SUBMISSION_PATH.stem}.7z" "{SUBMISSION_PATH.name}"


7-Zip (z) 26.04 (arm64) : Copyright (c) 1999-2026 Igor Pavlov : 2026-10-05
 64-bit arm_v:8.5-A locale=C.UTF-8 Threads:14 OPEN_MAX:1048575, ASM

Scanning the drive:
  0M Sca        1 file, 713013360 bytes (680 MiB)

Creating archive: submission_mlp_4out_emb_step.7z

Add new data to archive: 1 file, 713013360 bytes (680 MiB)

      0% + submission_mlp_4out_emb_step.cs                                        1% 1 + submission_mlp_4out_emb_step.cs                                          2% 1 + submission_mlp_4out_emb_step.cs                                          3% 1 + submission_mlp_4out_emb_step.cs                                          4% 1 + submission_mlp_4out_emb_step.cs                                          5% 1 + submission_mlp_4out_emb_step.cs                                          6% 1 + submission_mlp_4out_emb_step.cs                                          7% 1 + submission_mlp_4out_emb_step.cs                                          8% 1 + submission_mlp_4out_emb_

## Settings

- One output per meter type
- Building identity as a learned embedding of size 16
- 2 hidden layers of 64, ReLU, no BatchNorm or dropout
- Adam at lr 1e-3, divided by 10 at epochs 5 and 7, batch 4096, 10 epochs
